# Formatos de Entrada e Saída: papéis, chat templates e saídas estruturadas

**Tema:** Estruturação de papéis (System, User, Assistant) e garantia de respostas estruturadas (saídas em JSON / Structured Outputs).

**Conteúdo**

*Parte 1: papéis e chat templates*

1. Os três papéis: `system`, `user` e `assistant`
2. Chat templates (com um exemplo real)
3. Por que separar os papéis: privilégios e segurança

*Parte 2: saídas estruturadas*

4. Saídas estruturadas (Structured Outputs): JSON Schema e JSON Mode (FRI)
5. Qual tipo escolher?
6. NL-to-Format

## 1. Os três papéis

Dentro da engenharia de prompt, existe uma distinção de papéis, a qual é seguida, para que as interações com LLMs tenham maior controle, consistência e até desempenho em determinados contextos. Existem três papéis fundamentais: **system**, **user** e **assistant**.

Na prática, uma conversa é uma **lista de mensagens**, e cada mensagem tem um `role` (papel) e um `content` (conteúdo).

### 1.1 O papel `system`

O papel do system, ou system prompt, é definir um comportamento de alto nível para o modelo. Geralmente, é a primeira instrução lida pelo modelo, ajuda a estabelecer um contexto geral e as diretrizes que devem ser seguidas.

Alguns exemplos simples: “Você é um classificador de e-mails” ou “Você é um chatbot que sempre responde como pirata”.

### 1.2 O papel `user`

O papel do user, ou user prompt, é a entrada fornecida pelo usuário, como perguntas, comandos ou tarefas específicas para que a IA responda. Basicamente, os user prompts são a parte interativa com o LLM que representa o pedido ao modelo, os quais podem ter diversos estilos de instruções, como perguntas específicas, contextualização do problema e fornecimento de exemplos, conhecido como Few-shots.

In [ ]:
# Pergunta específica
user_pergunta = {
    "role": "user",
    "content": "Qual a capital da França?",
}

# Contextualização do problema
user_contexto = {
    "role": "user",
    "content": (
        "Contexto: estou montando uma planilha de gastos mensais.\n"
        "Tarefa: sugira 5 categorias de despesa."
    ),
}

# Fornecimento de exemplos (few-shot)
user_fewshot = {
    "role": "user",
    "content": (
        "Dê o antônimo de cada palavra.\n\n"
        "Palavra: quente\n"
        "Antônimo: frio\n\n"
        "Palavra: alto\n"
        "Antônimo: baixo\n\n"
        "Palavra: rápido\n"
        "Antônimo:"
    ),
}

for nome, msg in [("Pergunta específica", user_pergunta),
                  ("Contextualização", user_contexto),
                  ("Few-shot", user_fewshot)]:
    print(f"--- {nome} ---")
    print(msg["content"])
    print()

--- Pergunta específica ---
Qual a capital da França?

--- Contextualização ---
Contexto: estou montando uma planilha de gastos mensais.
Tarefa: sugira 5 categorias de despesa.

--- Few-shot ---
Dê o antônimo de cada palavra.

Palavra: quente
Antônimo: frio

Palavra: alto
Antônimo: baixo

Palavra: rápido
Antônimo:



### 1.3 O papel `assistant`

O papel do assistant é o ponto para designar a resposta gerada pela LLM, ou seja, é a indicação para a resposta da IA. Além disso, ele representa, para o modelo, o histórico da conversa, permitindo a inclusão das respostas anteriores, marcadas com assistant, para que o modelo compreenda o contexto, tópico que se conecta com o gerenciamento de contexto mais à frente.

In [ ]:
# Classificação de sentimento: as mensagens "assistant" são respostas anteriores
exemplo_assistant = [
    {"role": "system", "content": "Classifique o sentimento do texto como positivo ou negativo."},
    {"role": "user", "content": "Adorei o produto"},
    {"role": "assistant", "content": "positivo"},     # resposta anterior
    {"role": "user", "content": "Chegou quebrado"},
    {"role": "assistant", "content": "negativo"},     # resposta anterior
    {"role": "user", "content": "Entrega rápida e produto ótimo"},  # nova entrada
    {"role": "assistant", "content": ""},
]

for m in exemplo_assistant:
    print(f"[{m['role']:<9}] {m['content']}")

[system   ] Classifique o sentimento do texto como positivo ou negativo.
[user     ] Adorei o produto
[assistant] positivo
[user     ] Chegou quebrado
[assistant] negativo
[user     ] Entrega rápida e produto ótimo
[assistant] 


As duas mensagens `assistant` acima são respostas anteriores. O modelo as lê como histórico da conversa e como exemplos do padrão esperado, e a próxima resposta (a do `assistant`) tende a seguir esse padrão.

## 2. Chat templates

O modelo de IA ainda lê essa sequência de papéis como tokens, mas desta vez com tokens de controle para que o modelo entenda a estrutura requisitada, essa estrutura é chamada de **chat templates**. Na realidade, quando mensagens são trocadas com o modelo a instrunção do usuário é transformada em um prompt para o modelo de LLM entender, como delimitar início, fim e quem está falando (system, user, assistant). Desse modo, mesmo o usuário estruturando os papéis ou não, os LLMs utilizam esses papéis internamente.

Cada modelo usa um tipo de formato de chat diferente, ou seja, cada empresa provedora do modelo de IA exige que você escreva o código da API do jeito deles. No entanto, existem bibliotecas de abstrações que traduzem o código que você escreve para o padrão da API destino, embora não seja universal, então é sempre necessário verificar o chat format para cada modelo.

### Exemplo: chatbot pirata com uma LLM real


In [ ]:
%pip install -q transformers torch

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import logging as hf_logging
from huggingface_hub.utils import disable_progress_bars

hf_logging.set_verbosity_error()   # esconde o aviso de requisição sem token (HF_TOKEN)
disable_progress_bars()            # sem barras de progresso (o GitHub não renderiza)

MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

torch.manual_seed(42)  # respostas mais reprodutíveis

dispositivo = "cuda" if torch.cuda.is_available() else "cpu"
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
modelo = AutoModelForCausalLM.from_pretrained(MODEL_ID).to(dispositivo)
modelo.eval()

print(f"Modelo carregado em: {dispositivo}")

In [ ]:
def gerar_resposta(mensagens, max_novos_tokens=120, temperatura=0.7):
    """Recebe a lista de mensagens e devolve só o texto da resposta do modelo."""
    # lista de mensagens -> texto com os tokens de controle do chat template
    texto = tokenizer.apply_chat_template(
        mensagens, tokenize=False, add_generation_prompt=True
    )
    entradas = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(dispositivo)

    with torch.no_grad():
        saida = modelo.generate(
            **entradas,
            max_new_tokens=max_novos_tokens,
            do_sample=True,
            temperature=temperatura,
            top_p=0.9,
        )

    # decodifica apenas os tokens novos (sem repetir o prompt)
    novos = saida[0][entradas["input_ids"].shape[-1]:]
    return tokenizer.decode(novos, skip_special_tokens=True).strip()

###Exemplo: Chat Template
O chatbot respondendo como um Rei. Agora com `system` e `user`. Primeiro vemos o texto que o modelo recebe, depois a resposta.

In [ ]:
mensagens = [
    {
        "role": "system",
        "content": """
        Você deve seguir obrigatoriamente estas regras:
        1. Você é um rei.
        2. Sempre responda em português.
        3. Todas as suas respostas devem terminar exatamente com:
        "Agora, traga-me um copo de água!"
        """
    },
    {
        "role": "user",
        "content": "Quanto é 2 + 2?"
    }
]

print("=== Texto enviado ao modelo (após o chat template) ===")
print(tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True))

print("=== Resposta do assistant ===")
print(gerar_resposta(mensagens))

- Os marcadores impressos (como `<|im_start|>`) são específicos deste modelo. Outros modelos usam outros marcadores.
- O texto termina com o marcador do `assistant` aberto: é a deixa para o modelo continuar escrevendo a resposta a partir dali.
- O modelo é pequeno: a resposta pode ter erros e varia a cada execução. O objetivo é mostrar a estrutura, não a qualidade.

## 3. Por que separar os papéis: privilégios e segurança

Esta separação para a entrada surgiu com motivações e trouxe consequências. Um dos principais problemas era a falta de níveis de privilégio, o que gerava ataques de prompt injection, instruções maliciosas, e jailbreak, ato de contornar travas de segurança ultrapassando limites éticos. Dessa forma, com a criação dos privilégios, a autoridade foi dividida, sem conferir controle total ao usuário. Junto disso, foi atribuída ao usuário a opção de estruturar o chat template, enquanto o desenvolvedor no nível acima restringe até certo ponto a liberdade do usuário, mas entregando uma ferramenta que pode trazer consistência e otimização, enquanto mantém a segurança interna.

---

# Parte 2: saídas estruturadas (JSON / Structured Outputs)

## 4. Saídas Estruturadas (Structured Outputs)

Saídas estruturadas são uma maneira de obter respostas consistentes e/ou legíveis por máquina. O objetivo é fazer com que o LLM retorne a resposta seguindo um modelo (formato) predefinido. Assim, em vez de um texto livre que precisa ser analisado, é possível receber uma resposta dentro de uma estrutura conhecida, que pode ser usada diretamente na integração com diferentes aplicações.

O formato de saída mais comum é o **JSON**, mas existem outros, como:

- pedir que o modelo faça uma escolha entre opções fixas (ex.: positivo ou negativo);
- pedir a entrega em um formato específico (ex.: um e-mail);
- Tool Calling, que será abordado em outro tópico.

Aqui exploraremos o JSON, uma estrutura universal para a troca de dados entre sistemas. Com ele, é possível padronizar a saída para praticamente qualquer tipo de sistema, independentemente da linguagem de programação.


### 4.1 Tipos de saída em JSON

Existem dois tipos principais:

1. **JSON Schema**: o usuário impõe uma *decodificação com restrições*. A saída do LLM é limitada a um espaço de tokens predefinido, o que garante que o resultado seja um JSON válido e que siga rigorosamente o esquema informado, portanto sendo um modelo determinístico.
2. **JSON Mode**: tem uma abordagem mais relaxada e não restringe totalmente a saída. Em vez disso, o usuário apenas orienta que a resposta seja em JSON. Esse tipo também é conhecido como **Format-Restricting Instructions (FRI)** e não se limita ao JSON: é possível instruir outros formatos, como XML, ou qualquer saída de preferência do usuário.

### Preparação: reaproveitando o modelo da Seção 2

Os exemplos abaixo usam o **mesmo modelo da Seção 2** (`Qwen/Qwen2.5-1.5B-Instruct`), que já está carregado nas variáveis `tokenizer` e `modelo`. Não há download de outro modelo.

Para a decodificação com restrições, usamos a biblioteca **XGrammar**: a partir de um esquema JSON, ela bloqueia, a cada passo da geração, os tokens que quebrariam o esquema. Ela se conecta ao `generate` do Transformers por meio de um `LogitsProcessor`, que ajusta as probabilidades dos tokens antes de cada escolha.

In [ ]:
%pip install -q xgrammar

In [ ]:
import json
import xgrammar as xgr

avaliacao = "Comprei o fone há uma semana. O som é ótimo, mas a bateria dura bem menos do que o anunciado."

# O XGrammar precisa conhecer o vocabulário do modelo para saber quais tokens bloquear
info = xgr.TokenizerInfo.from_huggingface(tokenizer, vocab_size=modelo.config.vocab_size)
compilador = xgr.GrammarCompiler(info)


def gerar_json(mensagens, schema=None, max_novos_tokens=200):
    """Igual à gerar_resposta da Seção 2, mas determinística e com restrição opcional (schema)."""
    texto = tokenizer.apply_chat_template(mensagens, tokenize=False, add_generation_prompt=True)
    entradas = tokenizer(texto, return_tensors="pt", add_special_tokens=False).to(dispositivo)

    # Com schema, só os tokens permitidos por ele podem ser escolhidos
    restricao = None
    if schema:
        gramatica = compilador.compile_json_schema(json.dumps(schema))
        restricao = [xgr.contrib.hf.LogitsProcessor(gramatica)]   # um novo a cada geração

    saida = modelo.generate(
        **entradas,
        max_new_tokens=max_novos_tokens,
        do_sample=False,                            # sempre o token mais provável
        temperature=None, top_p=None, top_k=None,   # só valem com amostragem
        repetition_penalty=1.0,                     # chaves e aspas se repetem em JSON
        logits_processor=restricao,
    )
    novos = saida[0][entradas["input_ids"].shape[-1]:]
    return tokenizer.decode(novos, skip_special_tokens=True).strip()

### Exemplo 1: JSON Schema

Aqui definimos o esquema esperado e o passamos para `gerar_json` (`schema=schema`). Por baixo, o XGrammar compila o esquema em uma gramática e, a cada passo, bloqueia os tokens que levariam a uma saída fora dele, então a saída é obrigada a respeitar as chaves, os tipos e os valores permitidos (`enum`).

In [ ]:
schema = {
    "type": "object",
    "properties": {
        "sentimento": {"type": "string", "enum": ["positivo", "negativo", "neutro"]},
        "nota": {"type": "integer"},
        "resumo": {"type": "string"},
    },
    "required": ["sentimento", "nota", "resumo"],
}

mensagens = [
    {"role": "system", "content": "Você analisa avaliações de produtos. A nota vai de 1 a 5."},
    {"role": "user", "content": avaliacao},
]

resposta = gerar_json(mensagens, schema=schema)   # decodificação restrita ao esquema

dados = json.loads(resposta)
print(dados)
print(dados["sentimento"])   # o campo existe e o valor está dentro do enum

### Exemplo 2: JSON Mode (FRI)

Aqui o formato é pedido **apenas por instrução no prompt**: a decodificação não tem nenhuma restrição (sem `schema`). Nada garante que a saída seja um JSON válido nem que siga um esquema, pois as chaves e os valores dependem que o modelo siga as instruções. Por isso, o prompt precisa descrever a estrutura desejada e a resposta deve ser interpretada e validada com cuidado.

In [ ]:
mensagens = [
    {
        "role": "system",
        "content": (
            "Você analisa avaliações de produtos. "
            "Responda apenas em JSON, com as chaves: "
            "'sentimento' (positivo, negativo ou neutro), "
            "'nota' (inteiro de 1 a 5) e 'resumo' (texto curto)."
        ),
    },
    {"role": "user", "content": avaliacao},
]

resposta = gerar_json(mensagens)    # sem schema: o formato depende só do prompt

print("Resposta bruta do modelo:")
print(resposta)

# Modelos costumam envolver o JSON em ```json ... ```: remove essas marcações antes de ler
limpa = resposta.replace("```json", "").replace("```", "").strip()

try:
    dados = json.loads(limpa)
    print("\nJSON interpretado:", dados)
    # Diferente do JSON Schema, é preciso validar as chaves e os valores
    print("Campos esperados presentes:", {"sentimento", "nota", "resumo"}.issubset(dados.keys()))
    print("Sentimento válido:", dados.get("sentimento") in {"positivo", "negativo", "neutro"})
except (json.JSONDecodeError, AttributeError):
    print("\nA saída não pôde ser lida como um objeto JSON.")

## 5. Qual tipo escolher?

A escolha é bastante variável. Nenhum tipo é necessariamente melhor que o outro, depende do contexto da tarefa, do prompt e do modelo de LLM utilizado. No artigo de **Tam et al. (2024)**, que compara diferentes estratégias de restrição de formato em LLMs, os resultados mostram que em **tarefas de raciocínio**, como resolução de problemas matemáticos e manipulação de símbolos, o JSON Schema leva a resultados piores que o JSON Mode, pois a restrição prejudica o raciocínio do modelo. No entanto, em **tarefas de classificação**, como diagnósticos médicos e análise financeira, o JSON Schema se sobressai, porque a restrição reduz o erro de seleção, enquanto um formato mais flexível pode gerar distrações.


## 6. NL-to-Format

Vale destacar um tipo de saída bastante utilizado, o **NL-to-Format**. Ele consiste em um processo de duas etapas, primeiro instruir o LLM a responder em **linguagem natural**, sem nenhuma restrição, e então instruir o LLM a converter a resposta obtida para o formato desejado. Esta técnica busca separar o conteúdo gerado do formato restrito, preservando o desempenho da resposta livre, mas entregando uma saída estruturada. Embora pareça combinar as qualidades dos dois tipos anteriores, ela também tem pontos fracos, como a introdução de **erros durante a conversão**, o **custo computacional e de execução maior**, por exigir um prompt adicional e, como já dito, a melhor escolha continua dependendo do contexto.